# 作业 1：逻辑回归
欢迎来到本专项课程的第一周。你将学习逻辑回归。具体来说，你将实现用于推文情感分析的逻辑回归。给定一条推文，你需要判断它是积极情感还是消极情感。你将完成以下内容：

* 学习如何从文本中提取逻辑回归所需的特征
* 从零开始实现逻辑回归
* 将逻辑回归应用于自然语言处理任务
* 测试你的逻辑回归模型
* 进行错误分析

我们将使用一个推文数据集。希望你能达到 99% 以上的准确率。
运行下面的代码单元来加载所需的包。

## 导入函数和数据

In [ ]:
# run this cell to import nltk
import nltk
from os import getcwd

### 已导入的函数

下载本作业所需的数据。请查看 [twitter_samples 数据集文档](http://www.nltk.org/howto/twitter.html)。

* twitter_samples：如果你在本地电脑上运行此 notebook，需要使用以下命令下载：
```Python
nltk.download('twitter_samples')
```

* stopwords：如果你在本地电脑上运行此 notebook，需要使用以下命令下载：
```python
nltk.download('stopwords')
```

#### 导入 utils.py 文件中提供的一些辅助函数：
* `process_tweet()`：清洗文本，将其分词为独立单词，移除停用词，并将单词转换为词干。
* `build_freqs()`：统计'语料库'（全部推文集合）中每个单词与积极标签 '1' 或消极标签 '0' 关联的次数，然后构建 `freqs` 字典，其中每个键是一个 (单词, 标签) 元组，值是该单词在推文语料库中出现的频率计数。

In [39]:
# add folder, tmp2, from our local workspace containing pre-downloaded corpora files to nltk's data path
# this enables importing of these files without downloading it again when we refresh our workspace

filePath = f"{getcwd()}/../tmp2/"
nltk.data.path.append(filePath)

In [40]:
import numpy as np
import pandas as pd
from nltk.corpus import twitter_samples 

from utils import process_tweet, build_freqs

### 准备数据
* `twitter_samples` 包含 5000 条积极推文的子集、5000 条消极推文的子集，以及包含 10000 条推文的完整集合。
    * 如果你使用全部三个数据集，将会引入积极推文和消极推文的重复。
    * 你只需选择 5000 条积极推文和 5000 条消极推文。

In [16]:
# select the set of positive and negative tweets
all_positive_tweets = twitter_samples.strings('positive_tweets.json')
all_negative_tweets = twitter_samples.strings('negative_tweets.json')

* 训练集/测试集划分：20% 作为测试集，80% 作为训练集。


In [17]:
# split the data into two pieces, one for training and one for testing (validation set) 
test_pos = all_positive_tweets[4000:]
train_pos = all_positive_tweets[:4000]
test_neg = all_negative_tweets[4000:]
train_neg = all_negative_tweets[:4000]

train_x = train_pos + train_neg 
test_x = test_pos + test_neg

* 创建积极标签和消极标签的 numpy 数组。

In [18]:
# combine positive and negative labels
train_y = np.append(np.ones((len(train_pos), 1)), np.zeros((len(train_neg), 1)), axis=0)
test_y = np.append(np.ones((len(test_pos), 1)), np.zeros((len(test_neg), 1)), axis=0)

In [19]:
# Print the shape train and test sets
print("train_y.shape = " + str(train_y.shape))
print("test_y.shape = " + str(test_y.shape))

train_y.shape = (8000, 1)
test_y.shape = (2000, 1)


* 使用导入的 `build_freqs()` 函数创建频率字典。
    * 我们强烈建议你打开 `utils.py` 并阅读 `build_freqs()` 函数，以理解它的作用。
    * 要查看文件目录，请转到菜单并点击 File->Open。

```Python
    for y,tweet in zip(ys, tweets):
        for word in process_tweet(tweet):
            pair = (word, y)
            if pair in freqs:
                freqs[pair] += 1
            else:
                freqs[pair] = 1
```
* 注意外层 for 循环遍历每条推文，内层 for 循环遍历推文中的每个单词。
* `freqs` 字典就是正在构建的频率字典。
* 键是元组 (单词, 标签)，例如 ("happy",1) 或 ("happy",0)。每个键对应的值是单词 "happy" 与积极标签关联的次数，或与消极标签关联的次数。

In [20]:
# create frequency dictionary
freqs = build_freqs(train_x, train_y)

# check the output
print("type(freqs) = " + str(type(freqs)))
print("len(freqs) = " + str(len(freqs.keys())))

type(freqs) = <class 'dict'>
len(freqs) = 11346


#### 预期输出
```
type(freqs) = <class 'dict'>
len(freqs) = 11346
```

### 处理推文
给定的 `process_tweet()` 函数将推文分词为独立单词，移除停用词，并应用词干提取。

In [21]:
# test the function below
print('This is an example of a positive tweet: \n', train_x[0])
print('\nThis is an example of the processed version of the tweet: \n', process_tweet(train_x[0]))

This is an example of a positive tweet: 
 #FollowFriday @France_Inte @PKuchly57 @Milipol_Paris for being top engaged members in my community this week :)

This is an example of the processed version of the tweet: 
 ['followfriday', 'top', 'engag', 'member', 'commun', 'week', ':)']


#### 预期输出
```
This is an example of a positive tweet: 
 #FollowFriday @France_Inte @PKuchly57 @Milipol_Paris for being top engaged members in my community this week :)
 
This is an example of the processes version: 
 ['followfriday', 'top', 'engag', 'member', 'commun', 'week', ':)']
```

# 第 1 部分：逻辑回归


### 第 1.1 节：Sigmoid 函数
你将学习使用逻辑回归进行文本分类。
* Sigmoid 函数定义如下：

$$ h(z) = \frac{1}{1+\exp^{-z}} \tag{1}$$

它将输入 'z' 映射到 0 到 1 之间的值，因此可以将其视为概率。

<div style="width:image width px; font-size:100%; text-align:center;"><img src='../tmp2/sigmoid_plot.jpg' alt="alternate text" width="width" height="height" style="width:300px;height:200px;" /> 图 1 </div>

#### 说明：实现 sigmoid 函数
* 你需要让这个函数在 z 是标量或数组时都能正常工作。

<details>    
<summary>
    <font size="3" color="darkgreen"><b>提示</b></font>
</summary>
<p>
<ul>
    <li><a href="https://docs.scipy.org/doc/numpy/reference/generated/numpy.exp.html" > numpy.exp </a> </li>

</ul>
</p>



In [22]:
# UNQ_C1 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
def sigmoid(z): 
    '''
    Input:
        z: is the input (can be a scalar or an array)
    Output:
        h: the sigmoid of z
    '''
    
    ### START CODE HERE (REPLACE INSTANCES OF 'None' with your code) ###
    # calculate the sigmoid of z
    h = 1 / (1 + np.exp(-z))
    ### END CODE HERE ###
    
    return h

In [23]:
# Testing your function 
if (sigmoid(0) == 0.5):
    print('SUCCESS!')
else:
    print('Oops!')

if (sigmoid(4.92) == 0.9927537604041685):
    print('CORRECT!')
else:
    print('Oops again!')

SUCCESS!
CORRECT!


### 逻辑回归：回归 + sigmoid

逻辑回归在普通线性回归的基础上，对线性回归的输出应用 sigmoid 函数。

线性回归：
$$z = \theta_0 x_0 + \theta_1 x_1 + \theta_2 x_2 + ... \theta_N x_N$$
注意 $\theta$ 值是"权重"。如果你学习过深度学习专项课程，我们用 `w` 向量来表示权重。在本课程中，我们使用不同的变量 $\theta$ 来表示权重。

逻辑回归
$$ h(z) = \frac{1}{1+\exp^{-z}}$$
$$z = \theta_0 x_0 + \theta_1 x_1 + \theta_2 x_2 + ... \theta_N x_N$$
我们将 'z' 称为 'logits'（对数几率）。

### 第 1.2 节：代价函数和梯度

逻辑回归使用的代价函数是所有训练样本上对数损失的平均值：

$$J(\theta) = -\frac{1}{m} \sum_{i=1}^m y^{(i)}\log (h(z(\theta)^{(i)})) + (1-y^{(i)})\log (1-h(z(\theta)^{(i)}))\tag{5} $$
* $m$ 是训练样本的数量
* $y^{(i)}$ 是第 i 个训练样本的真实标签。
* $h(z(\theta)^{(i)})$ 是模型对第 i 个训练样本的预测。

单个训练样本的损失函数为
$$ Loss = -1 \times \left( y^{(i)}\log (h(z(\theta)^{(i)})) + (1-y^{(i)})\log (1-h(z(\theta)^{(i)})) \right)$$

* 所有的 $h$ 值都在 0 到 1 之间，因此对数将为负数。这就是为什么要在两个损失项之和前乘以 -1 的原因。
* 注意，当模型预测为 1（$h(z(\theta)) = 1$）且标签 $y$ 也为 1 时，该训练样本的损失为 0。
* 类似地，当模型预测为 0（$h(z(\theta)) = 0$）且真实标签也为 0 时，该训练样本的损失为 0。
* 然而，当模型预测接近 1（$h(z(\theta)) = 0.9999$）而标签为 0 时，对数损失的第二项将变成一个很大的负数，然后乘以整体系数 -1 转换为正的损失值。$-1 \times (1 - 0) \times log(1 - 0.9999) \approx 9.2$ 模型预测越接近 1，损失越大。

In [24]:
# verify that when the model predicts close to 1, but the actual label is 0, the loss is a large positive value
-1 * (1 - 0) * np.log(1 - 0.9999) # loss is about 9.2

9.210340371976294

* 同样地，如果模型预测接近 0（$h(z) = 0.0001$）但真实标签为 1，损失函数中的第一项将变成一个很大的数：$-1 \times log(0.0001) \approx 9.2$。预测越接近零，损失越大。

In [25]:
# verify that when the model predicts close to 0 but the actual label is 1, the loss is a large positive value
-1 * np.log(0.0001) # loss is about 9.2

9.210340371976182

#### 更新权重

要更新权重向量 $\theta$，你将应用梯度下降来迭代地改进模型的预测。
代价函数 $J$ 对其中一个权重 $\theta_j$ 的梯度为：

$$\nabla_{\theta_j}J(\theta) = \frac{1}{m} \sum_{i=1}^m(h^{(i)}-y^{(i)})x_j \tag{5}$$
* 'i' 是遍历所有 'm' 个训练样本的索引。
* 'j' 是权重 $\theta_j$ 的索引，因此 $x_j$ 是与权重 $\theta_j$ 关联的特征。

* 要更新权重 $\theta_j$，我们通过减去由 $\alpha$ 决定的梯度的一部分来调整它：
$$\theta_j = \theta_j - \alpha \times \nabla_{\theta_j}J(\theta) $$
* 学习率 $\alpha$ 是我们选择的一个值，用于控制单次更新的幅度。


## 说明：实现梯度下降函数
* 迭代次数 `num_iters` 是你使用整个训练集的次数。
* 每次迭代，你将使用所有训练样本（共有 `m` 个训练样本）以及所有特征来计算代价函数。
* 与其一次更新一个权重 $\theta_i$，我们可以同时更新列向量中的所有权重：
$$\mathbf{\theta} = \begin{pmatrix}
\theta_0
\\
\theta_1
\\ 
\theta_2 
\\ 
\vdots
\\ 
\theta_n
\end{pmatrix}$$
* $\mathbf{\theta}$ 的维度为 (n+1, 1)，其中 'n' 是特征数量，额外的一个元素是偏置项 $\theta_0$（注意对应的特征值 $\mathbf{x_0}$ 为 1）。
* 'logits' 'z' 通过特征矩阵 'x' 与权重向量 'theta' 相乘计算得到。$z = \mathbf{x}\mathbf{\theta}$
    * $\mathbf{x}$ 的维度为 (m, n+1)
    * $\mathbf{\theta}$：维度为 (n+1, 1)
    * $\mathbf{z}$：维度为 (m, 1)
* 预测 'h' 通过对 'z' 中的每个元素应用 sigmoid 计算得到：$h(z) = sigmoid(z)$，维度为 (m,1)。
* 代价函数 $J$ 通过取向量 'y' 和 'log(h)' 的点积计算。由于 'y' 和 'h' 都是列向量 (m,1)，将左侧向量转置，使行向量与列向量的矩阵乘法执行点积。
$$J = \frac{-1}{m} \times \left(\mathbf{y}^T \cdot log(\mathbf{h}) + \mathbf{(1-y)}^T \cdot log(\mathbf{1-h}) \right)$$
* theta 的更新也是向量化的。由于 $\mathbf{x}$ 的维度为 (m, n+1)，而 $\mathbf{h}$ 和 $\mathbf{y}$ 都是 (m, 1)，我们需要转置 $\mathbf{x}$ 并将其放在左侧以执行矩阵乘法，从而得到我们需要的 (n+1, 1) 结果：
$$\mathbf{\theta} = \mathbf{\theta} - \frac{\alpha}{m} \times \left( \mathbf{x}^T \cdot \left( \mathbf{h-y} \right) \right)$$

<details>    
<summary>
    <font size="3" color="darkgreen"><b>提示</b></font>
</summary>
<p>
<ul>
    <li>使用 np.dot 进行矩阵乘法。</li>
    <li>为确保分数 -1/m 是小数值，将分子或分母（或两者）转换类型，如 `float(1)`，或写 `1.` 表示 1 的浮点版本。</li>
</ul>
</p>



In [26]:
# UNQ_C2 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
def gradientDescent(x, y, theta, alpha, num_iters):
    '''
    Input:
        x: matrix of features which is (m,n+1)
        y: corresponding labels of the input matrix x, dimensions (m,1)
        theta: weight vector of dimension (n+1,1)
        alpha: learning rate
        num_iters: number of iterations you want to train your model for
    Output:
        J: the final cost
        theta: your final weight vector
    Hint: you might want to print the cost to make sure that it is going down.
    '''
    ### START CODE HERE (REPLACE INSTANCES OF 'None' with your code) ###
    # get 'm', the number of rows in matrix x
    m = x.shape[0]
    
    for i in range(0, num_iters):
        
        # get z, the dot product of x and theta
        z = np.dot(x, theta)
        
        # get the sigmoid of z
        h = sigmoid(z)
        
        # calculate the cost function
        J = -(np.dot(y.T, np.log(h)) + np.dot((1-y).T, np.log(1 - h)))/m

        # update the weights theta
        theta = theta - alpha * (np.dot(x.T, (h - y)))/m
        
    ### END CODE HERE ###
    J = float(J)
    return J, theta

In [27]:
# Check the function
# Construct a synthetic test case using numpy PRNG functions
np.random.seed(1)
# X input is 10 x 3 with ones for the bias terms
tmp_X = np.append(np.ones((10, 1)), np.random.rand(10, 2) * 2000, axis=1)
# Y Labels are 10 x 1
tmp_Y = (np.random.rand(10, 1) > 0.35).astype(float)

# Apply gradient descent
tmp_J, tmp_theta = gradientDescent(tmp_X, tmp_Y, np.zeros((3, 1)), 1e-8, 700)
print(f"The cost after training is {tmp_J:.8f}.")
print(f"The resulting vector of weights is {[round(t, 8) for t in np.squeeze(tmp_theta)]}")

The cost after training is 0.67094970.
The resulting vector of weights is [4.1e-07, 0.00035658, 7.309e-05]


#### 预期输出
```
The cost after training is 0.67094970.
The resulting vector of weights is [4.1e-07, 0.00035658, 7.309e-05]
```

## 第 2 部分：提取特征

* 给定推文列表，提取特征并将它们存储在一个矩阵中。你将提取两个特征。
    * 第一个特征是推文中积极单词的数量。
    * 第二个特征是推文中消极单词的数量。
* 然后在这些特征上训练你的逻辑回归分类器。
* 在验证集上测试分类器。

### 说明：实现 extract_features 函数。
* 该函数接收单条推文。
* 使用导入的 `process_tweet()` 函数处理推文，并保存推文单词列表。
* 遍历处理后单词列表中的每个单词
    * 对于每个单词，在 `freqs` 字典中查找该单词带有积极 '1' 标签时的计数。（查找键 (word, 1.0)）
    * 同样查找该单词与消极标签 '0' 关联时的计数。（查找键 (word, 0.0)。）

<details>    
<summary>
    <font size="3" color="darkgreen"><b>提示</b></font>
</summary>
<p>
<ul>
    <li>确保处理 (word, label) 键在字典中找不到的情况。</li>
    <li>在网上搜索关于使用 Python 字典 `.get()` 方法的提示。这里有一个<a href="https://www.programiz.com/python-programming/methods/dictionary/get" > 示例 </a></li>
</ul>
</p>

In [28]:
# UNQ_C3 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
def extract_features(tweet, freqs):
    '''
    Input: 
        tweet: a list of words for one tweet
        freqs: a dictionary corresponding to the frequencies of each tuple (word, label)
    Output: 
        x: a feature vector of dimension (1,3)
    '''
    # process_tweet tokenizes, stems, and removes stopwords
    word_l = process_tweet(tweet)
    
    # 3 elements in the form of a 1 x 3 vector
    x = np.zeros((1, 3)) 
    
    #bias term is set to 1
    x[0,0] = 1 
    
    ### START CODE HERE (REPLACE INSTANCES OF 'None' with your code) ###
    
    # loop through each word in the list of words
    for word in word_l:
        
        # increment the word count for the positive label 1
        x[0,1] += freqs.get((word, 1.0), 0)
        
        # increment the word count for the negative label 0
        x[0,2] += freqs.get((word, 0.0), 0)
        
    ### END CODE HERE ###
    assert(x.shape == (1, 3))
    return x

In [29]:
# Check your function

# test 1
# test on training data
tmp1 = extract_features(train_x[0], freqs)
print(tmp1)

[[1.00e+00 3.02e+03 6.10e+01]]


#### 预期输出
```
[[1.00e+00 3.02e+03 6.10e+01]]
```

In [30]:
# test 2:
# check for when the words are not in the freqs dictionary
tmp2 = extract_features('blorb bleeeeb bloooob', freqs)
print(tmp2)

[[1. 0. 0.]]


#### 预期输出
```
[[1. 0. 0.]]
```

## 第 3 部分：训练你的模型

训练模型：
* 将所有训练样本的特征堆叠成矩阵 `X`。
* 调用你在上面实现的 `gradientDescent`。

这部分内容已经为你提供。请阅读以理解其原理并运行代码单元。

In [31]:
# collect the features 'x' and stack them into a matrix 'X'
X = np.zeros((len(train_x), 3))
for i in range(len(train_x)):
    X[i, :]= extract_features(train_x[i], freqs)

# training labels corresponding to X
Y = train_y

# Apply gradient descent
J, theta = gradientDescent(X, Y, np.zeros((3, 1)), 1e-9, 1500)
print(f"The cost after training is {J:.8f}.")
print(f"The resulting vector of weights is {[round(t, 8) for t in np.squeeze(theta)]}")

The cost after training is 0.24216529.
The resulting vector of weights is [7e-08, 0.0005239, -0.00055517]


**预期输出**：

```
The cost after training is 0.24216529.
The resulting vector of weights is [7e-08, 0.0005239, -0.00055517]
```

# 第 4 部分：测试你的逻辑回归

现在是时候在模型未见过的新输入上测试你的逻辑回归函数了。

#### 说明：编写 `predict_tweet`
预测一条推文是积极的还是消极的。

* 给定一条推文，处理它，然后提取特征。
* 将模型学习到的权重应用于特征以获得 logits。
* 对 logits 应用 sigmoid 以获得预测（0 到 1 之间的值）。

$$y_{pred} = sigmoid(\mathbf{x} \cdot \theta)$$

In [32]:
# UNQ_C4 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
def predict_tweet(tweet, freqs, theta):
    '''
    Input: 
        tweet: a string
        freqs: a dictionary corresponding to the frequencies of each tuple (word, label)
        theta: (3,1) vector of weights
    Output: 
        y_pred: the probability of a tweet being positive or negative
    '''
    ### START CODE HERE (REPLACE INSTANCES OF 'None' with your code) ###
    
    # extract the features of the tweet and store it into x
    x = extract_features(tweet, freqs)
    
    # make the prediction using x and theta
    y_pred = sigmoid(np.dot(x, theta))
    
    ### END CODE HERE ###
    
    return y_pred

In [33]:
# Run this cell to test your function
for tweet in ['I am happy', 'I am bad', 'this movie should have been great.', 'great', 'great great', 'great great great', 'great great great great']:
    print( '%s -> %f' % (tweet, predict_tweet(tweet, freqs, theta)))

I am happy -> 0.518580
I am bad -> 0.494339
this movie should have been great. -> 0.515331
great -> 0.515464
great great -> 0.530898
great great great -> 0.546273
great great great great -> 0.561561


**预期输出**：
```
I am happy -> 0.518580
I am bad -> 0.494339
this movie should have been great. -> 0.515331
great -> 0.515464
great great -> 0.530898
great great great -> 0.546273
great great great great -> 0.561561
```

In [34]:
# Feel free to check the sentiment of your own tweet below
my_tweet = 'I am learning :)'
predict_tweet(my_tweet, freqs, theta)

array([[0.81636424]])

## 使用测试集检查性能
使用上面的训练集训练模型后，通过在测试集上测试来检查模型在真实、未见过的数据上的表现。

#### 说明：实现 `test_logistic_regression`
* 给定测试数据和训练好的模型权重，计算逻辑回归模型的准确率。
* 使用你的 `predict_tweet()` 函数对测试集中的每条推文进行预测。
* 如果预测 > 0.5，将模型的分类 `y_hat` 设为 1，否则设为 0。
* 当 `y_hat` 等于 `test_y` 时预测正确。将所有相等的实例求和并除以 `m`。

<details>    
<summary>
    <font size="3" color="darkgreen"><b>提示</b></font>
</summary>
<p>
<ul>
    <li>使用 np.asarray() 将列表转换为 numpy 数组</li>
    <li>使用 np.squeeze() 将 (m,1) 维数组变为 (m,) 数组</li>
</ul>
</p>

In [35]:
# UNQ_C5 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
def test_logistic_regression(test_x, test_y, freqs, theta):
    """
    Input: 
        test_x: a list of tweets
        test_y: (m, 1) vector with the corresponding labels for the list of tweets
        freqs: a dictionary with the frequency of each pair (or tuple)
        theta: weight vector of dimension (3, 1)
    Output: 
        accuracy: (# of tweets classified correctly) / (total # of tweets)
    """
    
    ### START CODE HERE (REPLACE INSTANCES OF 'None' with your code) ###
    
    # the list for storing predictions
    y_hat = []
    
    for tweet in test_x:
        # get the label prediction for the tweet
        y_pred = predict_tweet(tweet, freqs, theta)
        
        if y_pred > 0.5:
            # append 1.0 to the list
            y_hat.append(1.0)
        else:
            # append 0 to the list
            y_hat.append(0)

    # With the above implementation, y_hat is a list, but test_y is (m,1) array
    # convert both to one-dimensional arrays in order to compare them using the '==' operator
    accuracy = np.sum((np.array(y_hat)==test_y.flatten())!=0)/ len(y_hat)
    ### END CODE HERE ###
    
    return accuracy

In [36]:
tmp_accuracy = test_logistic_regression(test_x, test_y, freqs, theta)
print(f"Logistic regression model's accuracy = {tmp_accuracy:.4f}")

Logistic regression model's accuracy = 0.9950


#### 预期输出：
```0.9950```  
相当不错！

# 第 5 部分：错误分析

在这一部分，你将看到一些模型分类错误的推文。你认为为什么会发生这些分类错误？具体来说，你的模型会对哪种类型的推文分类错误？

In [37]:
# Some error analysis done for you
print('Label Predicted Tweet')
for x,y in zip(test_x,test_y):
    y_hat = predict_tweet(x, freqs, theta)

    if np.abs(y - (y_hat > 0.5)) > 0:
        print('THE TWEET IS:', x)
        print('THE PROCESSED TWEET IS:', process_tweet(x))
        print('%d\t%0.8f\t%s' % (y, y_hat, ' '.join(process_tweet(x)).encode('ascii', 'ignore')))

Label Predicted Tweet
THE TWEET IS: @jaredNOTsubway @iluvmariah @Bravotv Then that truly is a LATERAL move! Now, we all know the Queen Bee is UPWARD BOUND : ) #MovingOnUp
THE PROCESSED TWEET IS: ['truli', 'later', 'move', 'know', 'queen', 'bee', 'upward', 'bound', 'movingonup']
1	0.49996890	b'truli later move know queen bee upward bound movingonup'
THE TWEET IS: @MarkBreech Not sure it would be good thing 4 my bottom daring 2 say 2 Miss B but Im gonna be so stubborn on mouth soaping ! #NotHavingit :p
THE PROCESSED TWEET IS: ['sure', 'would', 'good', 'thing', '4', 'bottom', 'dare', '2', 'say', '2', 'miss', 'b', 'im', 'gonna', 'stubborn', 'mouth', 'soap', 'nothavingit', ':p']
1	0.48622857	b'sure would good thing 4 bottom dare 2 say 2 miss b im gonna stubborn mouth soap nothavingit :p'
THE TWEET IS: I'm playing Brain Dots : ) #BrainDots
http://t.co/UGQzOx0huu
THE PROCESSED TWEET IS: ["i'm", 'play', 'brain', 'dot', 'braindot']
1	0.48370665	b"i'm play brain dot braindot"
THE TWEET IS: I'm p

在本专项课程的后续内容中，我们将学习如何使用深度学习来提高预测性能。

# 第 6 部分：用你自己的推文进行预测

In [38]:
# Feel free to change the tweet below
my_tweet = 'This is a ridiculously bright movie. The plot was terrible and I was sad until the ending!'
print(process_tweet(my_tweet))
y_hat = predict_tweet(my_tweet, freqs, theta)
print(y_hat)
if y_hat > 0.5:
    print('Positive sentiment')
else: 
    print('Negative sentiment')

['ridicul', 'bright', 'movi', 'plot', 'terribl', 'sad', 'end']
[[0.48139087]]
Negative sentiment


In [ ]:
import inspect
peinr(inspect.getfile(nlpk.))